In [2]:
# ! uv pip install google
! uv pip install google-generativeai

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Resolved 28 packages in 138ms                                        
Uninstalled 2 packages in 1ms
Installed 2 packages in 13ms                                
 - grpcio-status==1.76.0
 + grpcio-status==1.71.2
 - protobuf==6.33.3
 + protobuf==5.29.6


In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai
# Instead of: import openai
import google.generativeai as genai
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

2026-06-25 14:34:03.780395654 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/tmp/ipykernel_611512/3916736685.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### Read the sampled dataset with Amazon inventory data

In [4]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [5]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[],169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...",[],24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN


In [6]:
list(df_items["features"].items())[0]

(0,
 ['【Fast Charging Cord】These USB C cables provide up to a 3A charging current to greatly shorten the charging time, meets QC2.0 /3.0 fast charging protocol,Incredibly charge your phone from 0 to 80% in 50 minute. 480Mbps (40-60M/s) ultra fast data transmission, which leads to a faster data sync.(Note:Cables support fast charging,but require a USB-A QC3.0/QC2.0/AFC charger)',
  '【Universal Compatibility】The USB C Charger Cable is compatible with Samsung Galaxy S20 / S10 / S9 / S8+ / S8 / A02s / A03s,A12 A20 A21 A22 A23 A31 A32 A33 A41 A42 A50 A52 A52s 5G A71 A72 A73,M11 M21 M31 M51,M12 M22 M32 M52,iPad Pro 2018 / 2020,Sony Xperia XZ/X Compact/L1 / XZs / XA1 / X Premium, HTC 10 LG G5 G6,OnePlus 5T / 6T, Lumia 950 / 950XL,Huawei P9 P9 Plus P10 P10 Plus Honor Mate 9 Mate 9 pro Mate 10 pro Mate 10 lite and more',
  '【Premium Workmanship】Unique increased friction design allows you to easily unplug the cable from your charger,combine 250d bulletproof fiber core to build a cable so durable

In [7]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51G07yWoOBL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51G07yWoOBL.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/611AVJMH+JL._SL1200_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41c+40oKkKL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41c+40oKkKL.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/61ihhPW7VCL._SL1200_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51y1YnwiUZL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51y1YnwiUZL.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/61UkcVETKcL._SL1200_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41Nvr++q39L._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41Nvr++q39L.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.c

### Preprocess title and features

In [8]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [9]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [10]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [11]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51G07yWoOB...
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41bOA5-ogW...
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,USB C Docking Station Dual Monitor for MacBook...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN,https://m.media-amazon.com/images/I/416IzmVKiC...
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[2023 Upgraded] Telescopes for Adults Astronom...,169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41wO4J3TT0...
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...","Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41mwlYqT5p...


In [12]:
list(df_items["description"].items())[0]

(0,
 "RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB Type C Fast Charging Cord - Nylon Braided USB C Charger Cable for Galaxy A20/A50/S10/S9/S8+/S8, iPad Pro 2018, Sony XZ, HTC 10, OnePlus 5T, Huawei P9 etc. 【Fast Charging Cord】These USB C cables provide up to a 3A charging current to greatly shorten the charging time, meets QC2.0 /3.0 fast charging protocol,Incredibly charge your phone from 0 to 80% in 50 minute. 480Mbps (40-60M/s) ultra fast data transmission, which leads to a faster data sync.(Note:Cables support fast charging,but require a USB-A QC3.0/QC2.0/AFC charger) 【Universal Compatibility】The USB C Charger Cable is compatible with Samsung Galaxy S20 / S10 / S9 / S8+ / S8 / A02s / A03s,A12 A20 A21 A22 A23 A31 A32 A33 A41 A42 A50 A52 A52s 5G A71 A72 A73,M11 M21 M31 M51,M12 M22 M32 M52,iPad Pro 2018 / 2020,Sony Xperia XZ/X Compact/L1 / XZs / XA1 / X Premium, HTC 10 LG G5 G6,OnePlus 5T / 6T, Lumia 950 / 950XL,Huawei P9 P9 Plus P10 P10 Plus Honor Mate 9 Mate 9 pro Mate 10 pro Mate 1

### Sample 50 items from the dataset

In [13]:
df_sample = df_items.sample(50, random_state=42)

In [14]:
len(df_sample)

50

In [15]:
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [16]:
data_to_embed

[{'description': 'KEEPRO Pencil 2nd Generation for iPad, Magnetic Wireless Charge Tilt Sensitivity Palm Rejection Active Pen for Apple iPad Pro 11" 4/3/2/1, iPad Pro 12.9" 6/5/4/3, iPad Air 4/5, iPad Mini 6 [Compatibility]- ONLY compatible with iPad mini (6th generation), iPad Air (4th and 5th generation), iPad Pro 12.9-inch (3rd, 4th, 5th and 6th generation), iPad Pro 11-inch (1st, 2nd, 3rd and 4th generation), check and confirm your device before place the order (Note: If the pen doesn\'t charge, fully charge your iPad first then try charging the pen again) [Charging and Pairs Magnetically]- Charges wirelessly, attaches and pairs magnetically to the compatible iPad, this pen is a preferable alternative to the Apple Pencil 2nd Generation [Tilt Sensitivity & Pixel Precision]- Pixel-perfect precision and industry-leading low latency with tilt sensitivity making drawing, sketching, coloring, taking notes, and marking up PDFs, as easy and natural as a real pencil [Native Palm Rejection]- 

### Define the embedding function

In [20]:
# response = openai.embeddings.create(
#     input="Random text",
#     model="text-embedding-3-small",
# )

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# len(response.data[0].embedding)

In [ ]:
# def get_embedding(text, model="text-embedding-3-small"):
#     response = openai.embeddings.create(
#         input=text,
#         model=model,
#     )
#     return response.data[0].embedding

In [ ]:
# get_embedding("Hi")

In [17]:
# Load variables from .env into the environment
load_dotenv()

# Check if it worked
print("GOOGLE_API_KEY is set:", bool(os.getenv("GOOGLE_API_KEY")))

GOOGLE_API_KEY is set: True


In [18]:
from google import genai

gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=["You are a helpful assistant",
              "can you print 15 random words?"]
)

print(response.text)

Okay, here are 15 random words:

1.  **Mountain**
2.  **Whisper**
3.  **Laptop**
4.  **Curtain**
5.  **Journey**
6.  **Biscuit**
7.  **Shadow**
8.  **Vibrant**
9.  **Ocean**
10. **Quirk**
11. **Symphony**
12. **Puzzle**
13. **Giggle**
14. **Horizon**
15. **Velvet**


In [19]:
# def get_embedding(text, model="models/text-embedding-004", task_type="RETRIEVAL_DOCUMENT"):
#     """
#     task_type="RETRIEVAL_DOCUMENT" is best practice for data going INTO your database.
#     We will use task_type="RETRIEVAL_QUERY" when searching.
#     """
#     response = gemini_client.embed_content(
#         model=model,
#         content=text,
#         task_type=task_type
#     )
#     return response['embedding']

In [20]:
get_embedding("Hi")

AttributeError: 'Client' object has no attribute 'embed_content'

In [ ]:
# # 🔴 FIXED: Use genai.embed_content directly, not gemini_client.embed_content
# def get_embedding(text, model="models/text-embedding-004", task_type="RETRIEVAL_DOCUMENT"):
#     """
#     Using the module-level function directly. 
#     Ensure genai.configure(api_key=...) was called previously.
#     """
#     response = genai.embed_content(
#         model=model,
#         content=text,
#         task_type=task_type
#     )
#     return response['embedding']



In [21]:
# def get_embedding(text, model="models/text-embedding-004", task_type="RETRIEVAL_DOCUMENT"):
#     """
#     Embeds text using Gemini's embedding model.
#     task_type="RETRIEVAL_DOCUMENT" for indexing, "RETRIEVAL_QUERY" for searching.
#     """
#     response = gemini_client.models.embed_content(   # <-- Correct method
#         model=model,
#         contents=[text],                    # Pass a list, even for single text
#         config=types.EmbedContentConfig(
#             task_type=task_type
#         )
#     )
#     # The response returns an object with an 'embeddings' list
#     return response.embeddings[0].values 

In [22]:
get_embedding("Hi")

NameError: name 'types' is not defined

In [24]:
# # # Initialize the Gemini client
# # gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# def get_embedding(text, model="models/text-embedding-004", task_type="RETRIEVAL_DOCUMENT"):
#     """
#     Embeds text using Gemini's embedding model.
#     task_type="RETRIEVAL_DOCUMENT" for indexing, "RETRIEVAL_QUERY" for searching.
#     """
#     response = gemini_client.models.embed_content(
#         model=model,
#         contents=[text],                     # Must be a list
#         config=types.EmbedContentConfig(     # <-- Now 'types' is defined
#             task_type=task_type
#         )
#     )
#     # Extract the embedding vector (list of floats, length = 768)
#     return response.embeddings[0].values

In [26]:
# get_embedding("Hi")

NameError: name 'types' is not defined

In [27]:
# 2. Embedding function (Using the most compatible syntax)
def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    """
    Embeds text using Gemini.
    task_type options: "RETRIEVAL_DOCUMENT" (indexing), "RETRIEVAL_QUERY" (searching)
    """
    model = "models/text-embedding-004"
    response = genai.embed_content(
        model=model,
        content=text,
        task_type=task_type
    )
    # The response from genai.embed_content is a dictionary
    return response['embedding']

In [28]:
get_embedding("Hi")

AttributeError: module 'google.genai' has no attribute 'embed_content'

In [29]:
def get_embedding(text, model="models/text-embedding-004"):
    response = gemini_client.models.embed_content(
        model=model,
        contents=[text]
    )
    return response.embeddings[0].values

In [30]:
get_embedding("Hi")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [22]:
def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    """
    Embeds text using the new google-genai SDK.
    task_type options: "RETRIEVAL_DOCUMENT" (indexing), "RETRIEVAL_QUERY" (searching)
    """
    # Note: Use just the model name, NO 'models/' prefix
    response = gemini_client.models.embed_content(
        model="text-embedding-004",
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type
        )
    )
    # The new SDK stores the vector in .embeddings[0].values
    return response.embeddings[0].values

In [20]:
get_embedding("Hi")

NameError: name 'client' is not defined

In [23]:
get_embedding("Hi")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [21]:
import os
from dotenv import load_dotenv
from google import genai

# Load API key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

# Initialize client
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

def get_embedding(text, model="models/embedding-001"):
    """
    Generates an embedding vector for the given text using Gemini's embedding-001.
    Returns a list of 768 floats.
    """
    response = gemini_client.models.embed_content(
        model=model,
        contents=[text]
    )
    return response.embeddings[0].values

# Quick test
if __name__ == "__main__":
    test_vec = get_embedding("Hello world")
    print(f"✅ Embedding generated! Length: {len(test_vec)}")  # 768
    print(f"First 5 values: {test_vec[:5]}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/embedding-001 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [24]:
# 3. Define the embedding function
def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    """
    Generates an embedding vector for the given text using Gemini's current model.
    """
    # 🔴 THE FIX: We must use the new model names! 
    # 'text-embedding-004' is dead and will cause a 404 error.
    response = gemini_client.models.embed_content(
        model="gemini-embedding-001", 
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type
        )
    )
    
    # The new SDK stores the vector inside the .values property of the embedding object
    return response.embeddings[0].values

# 4. Quick test
if __name__ == "__main__":
    print("Testing the embedding model...")
    test_vec = get_embedding("Hello world")
    
    print(f"✅ Success! Embedding generated.")
    print(f"Vector Length: {len(test_vec)}")
    print(f"First 5 values: {test_vec[:5]}")

Testing the embedding model...
✅ Success! Embedding generated.
Vector Length: 3072
First 5 values: [-0.026189324, 0.003135996, 0.01789595, -0.08768083, -0.0034239523]


In [25]:
get_embedding("Hi")

[-0.025007993,
 0.0031056749,
 0.006004557,
 -0.09255994,
 -0.010213805,
 0.009807502,
 -0.010265805,
 -0.012374833,
 -0.00056642323,
 -0.003342203,
 -0.0063933926,
 -0.0055228383,
 0.0052326084,
 0.0027167615,
 0.16389672,
 -0.012560059,
 -0.003093739,
 0.002989793,
 -0.000113608614,
 -0.010857533,
 -0.007634946,
 0.001932304,
 0.003503979,
 -0.0023980108,
 0.00090495305,
 0.0015144452,
 0.011442748,
 0.003051497,
 0.026757432,
 0.0107218055,
 -0.00036605142,
 -0.01048353,
 -0.0065046055,
 0.022761269,
 0.015756652,
 0.011479514,
 0.01307695,
 0.007838313,
 -0.012013674,
 0.0019863506,
 0.006653037,
 0.00010481909,
 -0.008352003,
 -0.008428091,
 -0.00036256417,
 0.0004092497,
 0.0040981714,
 -0.0072130207,
 0.0123988595,
 0.026668983,
 0.004204142,
 0.007496446,
 -0.025651637,
 -0.2843022,
 -0.016988542,
 -0.003793151,
 0.003937319,
 0.01064169,
 0.0049295863,
 -0.017635727,
 -0.004012328,
 0.00045201505,
 -0.02238965,
 -0.038184945,
 0.005280571,
 0.005075113,
 0.002598816,
 0.015735

### Create Qdrant collection

In [26]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [27]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

ResponseHandlingException: [Errno 111] Connection refused

### Embed data

##### Test

In [ ]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text": "Test text",
        "model": "text-embedding-3-small",
    },
)

In [ ]:
pointstruct

### Amazon data

In [ ]:
pointstructs = []
for i, data in enumerate(data_to_embed):
    embediing = get_embedding(data["description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embediing,
            payload=data,
        )
    )

In [ ]:
pointstructs

In [ ]:
len(pointstructs)

### Write embedded data to Qdrant

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",
    wait=True,
    points=pointstructs,
)

### Define a function for data retrieval

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )
    return results

### Test retrieval

In [ ]:
retrieve_data("What kind of charging cords do you offer?", k=10).points